# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prakritibhandari07/FlyRank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Method:** Logistic Regression first, then Random Forest for comparison.

**Why it fits:** The question is yes/no with an observed label (`is_declining_label`), which
the toolkit maps directly to "Logistic Regression, then Random Forest — readable → stronger."
My baseline is a ranking rule (a score used to prioritize a review queue), so the honest
comparison metric is precision@K, not accuracy — of the top K pages either method flags,
how many are actually declining? I start with Logistic Regression because its coefficients are
directly readable (same spirit as the baseline's transparent rule), then check whether Random
Forest earns its added complexity by actually beating it at the K values that matter, per the
skill's warning not to reward complexity alone.

In [1]:
!git clone https://github.com/Prakritibhandari07/FlyRank-ml-internship.git
%cd FlyRank-ml-internship

import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"Rows: {len(df)}, base rate: {df['is_declining_label'].mean():.3f}")

Cloning into 'FlyRank-ml-internship'...
remote: Enumerating objects: 159, done.
remote: Counting objects: 100% (159/159), done.
remote: Compressing objects: 100% (115/115), done.
remote: Total 159 (delta 64), reused 93 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (159/159), 1.88 MiB | 4.43 MiB/s, done.
Resolving deltas: 100% (64/64), done.
/content/FlyRank-ml-internship
Rows: 30000, base rate: 0.542


## 2. Split design

Split: Grouped by client_id, 75/25 train/test using GroupShuffleSplit with a fixed seed.

Why this is honest:  
Rows from the same client share publishing batches. A random row-level split would leak near-duplicate pages into both train and test, letting the model memorize client-specific patterns. Grouping ensures each client’s rows are entirely in train or test, so evaluation is on unseen clients — the real generalization test.

In [2]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print(f"Train: {len(train_df)} rows, {train_df['client_id'].nunique()} clients")
print(f"Test:  {len(test_df)} rows, {test_df['client_id'].nunique()} clients")
overlap = set(train_df["client_id"]) & set(test_df["client_id"])
print(f"Client overlap between train/test: {len(overlap)} (should be 0)")

Train: 22885 rows, 24 clients
Test:  7115 rows, 8 clients
Client overlap between train/test: 0 (should be 0)


## 3. Train + compare vs my baseline

Setup:

Same split, same metric (precision@K), same label as baseline.

Baseline score is recomputed identically to Week‑4.

Models are evaluated only on the test set.

Forbidden features (label-derived or future-window) are excluded: trend_direction, trend_pct, is_declining_label, impressions_last_30d, impressions_prev_30d, plus IDs.

Features used:

Numeric: days_since_last_update, content_age_days, impressions_90d, clicks_90d, pageviews_90d, sessions_90d, ctr, avg_position, engagement_rate, scroll_rate, ai_traffic_pct, search_volume, competition, cpc, word_count, char_count.

Categorical: position_tier, freshness_tier, content_type, main_intent, competition_level, age_tier.

Missingness flags: added for all NaN‑prone columns (search_volume, competition, cpc, word_count, char_count, engagement_rate, scroll_rate, ai_traffic_pct).

Models trained:

Logistic Regression (scaled features).

Random Forest (depth‑limited, 300 trees).

Compared against baseline rule.

Metric: Precision@K at K = 20, 50, 100, 500.

In [4]:
# --- Recreate the Week-4 baseline rule, identically ------------------------------------
def baseline_score(frame):
    stale = frame["freshness_tier"].isin(["91-180", "181+"]).astype(int)
    measurable = ((frame["impressions_90d"] >= 100) & (frame["sessions_90d"] > 0)).astype(int)
    tier_median_ctr = frame.groupby("position_tier", observed=True)["ctr"].transform("median")
    ctr_underperform = np.where(measurable == 1, (frame["ctr"] < 0.7 * tier_median_ctr).astype(int), 0)
    return stale * measurable * ctr_underperform * frame["impressions_90d"]

df["baseline_score"] = baseline_score(df)
test_df = df.iloc[test_idx].copy()

FORBIDDEN = ["trend_direction", "trend_pct", "is_declining_label",
             "impressions_last_30d", "impressions_prev_30d",
             "content_id", "client_id", "baseline_score"]

numeric_features = [
    "days_since_last_update", "content_age_days", "impressions_90d", "clicks_90d",
    "pageviews_90d", "sessions_90d", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct", "search_volume", "competition", "cpc",
    "word_count", "char_count"
]
categorical_features = ["position_tier", "freshness_tier", "content_type", "main_intent",
                         "competition_level", "age_tier"]

# Every numeric column that can carry systematic missingness gets a has_ flag before fillna —
# this now covers ALL NaN-prone columns (including engagement_rate, scroll_rate, ai_traffic_pct),
# not just the keyword-context ones.
nan_prone_cols = ["search_volume", "competition", "cpc", "word_count", "char_count",
                   "engagement_rate", "scroll_rate", "ai_traffic_pct"]

work = df.copy()
has_flag_cols = []
for col in nan_prone_cols:
    flag_col = f"has_{col}"
    work[flag_col] = work[col].notna().astype(int)
    work[col] = work[col].fillna(0)
    has_flag_cols.append(flag_col)

for col in categorical_features:
    work[col] = work[col].fillna("unknown")

feature_cols = numeric_features + has_flag_cols
X = pd.get_dummies(work[feature_cols + categorical_features], columns=categorical_features, drop_first=True)
y = work["is_declining_label"]

# Safety check — confirm no NaN slipped through before training
assert X.isna().sum().sum() == 0, f"NaN still present in: {X.columns[X.isna().any()].tolist()}"
print(f"Feature matrix: {X.shape}, NaN check passed")

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Train models ------------------------------------------------------------------------
logreg = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
logreg.fit(X_train_scaled, y_train)
logreg_scores = logreg.predict_proba(X_test_scaled)[:, 1]

rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=RANDOM_SEED, n_jobs=-1)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

baseline_test_scores = test_df["baseline_score"].values
y_test_arr = y_test.values

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate_test = y_test_arr.mean()
Ks = [20, 50, 100, 500]

comparison = pd.DataFrame({
    "K": Ks,
    "base_rate": [base_rate_test] * len(Ks),
    "baseline_precision@K": [precision_at_k(baseline_test_scores, y_test_arr, k) for k in Ks],
    "logreg_precision@K":   [precision_at_k(logreg_scores, y_test_arr, k) for k in Ks],
    "random_forest_precision@K": [precision_at_k(rf_scores, y_test_arr, k) for k in Ks],
})
print(f"Test set base rate: {base_rate_test:.3f} (n={len(y_test_arr)})\n")
comparison

Feature matrix: (30000, 43), NaN check passed
Test set base rate: 0.517 (n=7115)



,K,base_rate,baseline_precision@K,logreg_precision@K,random_forest_precision@K
0,20,0.516514,0.450,0.500,0.500
1,50,0.516514,0.540,0.540,0.540
2,100,0.516514,0.510,0.560,0.570
3,500,0.516514,0.544,0.556,0.584


## 4. Errors and interpretation

Top features: List the top 2–3 from Logistic Regression and Random Forest. Check if they agree. Confirm whether the top feature makes sense or looks suspiciously perfect (possible leakage).

Where wrong: Summarize false positives by position_tier / content_type. Look for systematic over‑flagging (like baseline’s weakness on page_3_5).

Three wrong cases: For each, one sentence explaining what the model saw that made it flag the page, and why that was actually wrong.

In [5]:
# --- Recreate the Week-4 baseline rule, identically ------------------------------------
def baseline_score(frame):
    stale = frame["freshness_tier"].isin(["91-180", "181+"]).astype(int)
    measurable = ((frame["impressions_90d"] >= 100) & (frame["sessions_90d"] > 0)).astype(int)
    tier_median_ctr = frame.groupby("position_tier", observed=True)["ctr"].transform("median")
    ctr_underperform = np.where(measurable == 1, (frame["ctr"] < 0.7 * tier_median_ctr).astype(int), 0)
    return stale * measurable * ctr_underperform * frame["impressions_90d"]

df["baseline_score"] = baseline_score(df)
test_df = df.iloc[test_idx].copy()

FORBIDDEN = ["trend_direction", "trend_pct", "is_declining_label",
             "impressions_last_30d", "impressions_prev_30d",
             "content_id", "client_id", "baseline_score"]

numeric_features = [
    "days_since_last_update", "content_age_days", "impressions_90d", "clicks_90d",
    "pageviews_90d", "sessions_90d", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct", "search_volume", "competition", "cpc",
    "word_count", "char_count"
]
categorical_features = ["position_tier", "freshness_tier", "content_type", "main_intent",
                         "competition_level", "age_tier"]

# Every numeric column that can carry systematic missingness gets a has_ flag before fillna,
# so a blind fillna(0) doesn't silently encode content_type (or the zero-session/zero-pageview
# case) into the features. This covers all NaN-prone columns per the data dictionary, not just
# the keyword-context ones.
nan_prone_cols = ["search_volume", "competition", "cpc", "word_count", "char_count",
                   "engagement_rate", "scroll_rate", "ai_traffic_pct"]

work = df.copy()
has_flag_cols = []
for col in nan_prone_cols:
    flag_col = f"has_{col}"
    work[flag_col] = work[col].notna().astype(int)
    work[col] = work[col].fillna(0)
    has_flag_cols.append(flag_col)

for col in categorical_features:
    work[col] = work[col].fillna("unknown")

feature_cols = numeric_features + has_flag_cols
X = pd.get_dummies(work[feature_cols + categorical_features], columns=categorical_features, drop_first=True)
y = work["is_declining_label"]

# Safety check — confirm no NaN slipped through before training
assert X.isna().sum().sum() == 0, f"NaN still present in: {X.columns[X.isna().any()].tolist()}"
print(f"Feature matrix: {X.shape}, NaN check passed")

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Train models ------------------------------------------------------------------------
logreg = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
logreg.fit(X_train_scaled, y_train)
logreg_scores = logreg.predict_proba(X_test_scaled)[:, 1]

rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=RANDOM_SEED, n_jobs=-1)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

baseline_test_scores = test_df["baseline_score"].values
y_test_arr = y_test.values

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate_test = y_test_arr.mean()
Ks = [20, 50, 100, 500]

comparison = pd.DataFrame({
    "K": Ks,
    "base_rate": [base_rate_test] * len(Ks),
    "baseline_precision@K": [precision_at_k(baseline_test_scores, y_test_arr, k) for k in Ks],
    "logreg_precision@K":   [precision_at_k(logreg_scores, y_test_arr, k) for k in Ks],
    "random_forest_precision@K": [precision_at_k(rf_scores, y_test_arr, k) for k in Ks],
})
print(f"Test set base rate: {base_rate_test:.3f} (n={len(y_test_arr)})\n")
comparison

Feature matrix: (30000, 43), NaN check passed
Test set base rate: 0.517 (n=7115)



,K,base_rate,baseline_precision@K,logreg_precision@K,random_forest_precision@K
0,20,0.516514,0.450,0.500,0.500
1,50,0.516514,0.540,0.540,0.540
2,100,0.516514,0.510,0.560,0.570
3,500,0.516514,0.544,0.556,0.584


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.